In [1]:
%reload_ext autoreload
%autoreload 2

import synapseclient
from pyarrow import fs
import pyarrow.parquet as pq
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from data_utils import *
from read_utils import *
import numpy as np

#######################################################
# Change this to the synapse Id of the Parquet folder
parquet_dir_id = 'syn52506069'
#######################################################

syn = synapseclient.login()


Welcome, yj167000!




UPGRADE AVAILABLE

A more recent version of the Synapse Client (3.1.1) is available. Your version (2.7.2) can be upgraded by typing:
    pip install --upgrade synapseclient

Python Synapse Client version 3.1.1 release notes

https://python-docs.synapse.org/build/html/news.html



In [2]:
# Get a storage token
token = syn.get_sts_storage_token(parquet_dir_id, permission="read_only")

# Connect to S3 using the token
s3 = fs.S3FileSystem(access_key=token['accessKeyId'], 
                     secret_key = token['secretAccessKey'],
                     session_token = token['sessionToken'])

bucket_path = token['bucket']+'/'+token['baseKey']+'/'

# List archive folders
archive_folders = s3.get_file_info(fs.FileSelector(bucket_path, recursive=False))
archive_paths = [x.path for x in archive_folders if "owner" not in x.path]

# List datasets in a specific archive
dataset_list = s3.get_file_info(fs.FileSelector(archive_paths[-1], recursive=False))
dataset_paths = [x.path for x in dataset_list if "owner" not in x.path]
data_path_names = [item.replace('recover-main-project/main/archive/2023-09-21/', '') for item in dataset_paths]
print(data_path_names)

# Read a dataset into a Pandas DataFrame
dataset = pq.ParquetDataset(dataset_paths[0], filesystem=s3)
df = dataset.read().to_pandas()
p_id = df.iloc[:,0]
p_id

['recover-main-project/main/archive/2023-11-10/dataset_enrolledparticipants', 'recover-main-project/main/archive/2023-11-10/dataset_enrolledparticipants_customfields_symptoms', 'recover-main-project/main/archive/2023-11-10/dataset_enrolledparticipants_customfields_treatments', 'recover-main-project/main/archive/2023-11-10/dataset_fitbitactivitylogs', 'recover-main-project/main/archive/2023-11-10/dataset_fitbitdailydata', 'recover-main-project/main/archive/2023-11-10/dataset_fitbitdevices', 'recover-main-project/main/archive/2023-11-10/dataset_fitbitintradaycombined', 'recover-main-project/main/archive/2023-11-10/dataset_fitbitrestingheartrates', 'recover-main-project/main/archive/2023-11-10/dataset_fitbitsleeplogs', 'recover-main-project/main/archive/2023-11-10/dataset_fitbitsleeplogs_sleeplogdetails', 'recover-main-project/main/archive/2023-11-10/dataset_googlefitsamples', 'recover-main-project/main/archive/2023-11-10/dataset_healthkitv2activitysummaries', 'recover-main-project/main/a

0       RA11303-00148
1       RA11306-00173
2       RA11601-00666
3       RA11601-00830
4       RA11703-00158
            ...      
4002    RP25006-00081
4003    RP25702-00183
4004    RP25702-00001
4005    RP25601-00275
4006    RP25702-00069
Name: ParticipantIdentifier, Length: 4007, dtype: object

In [3]:
df = get_dataset_df_str(dataset_paths, "dataset_fitbitdailydata", s3)
df_act_logs = get_dataset_df_str(dataset_paths, "dataset_fitbitactivitylogs", s3)
# df_intraday = get_dataset_df_str(dataset_paths, "dataset_fitbitintradaycombined", s3)
df_rhr = get_dataset_df_str(dataset_paths, "dataset_fitbitrestingheartrates", s3)
df_devices = get_dataset_df_str(dataset_paths, "dataset_fitbitdevices", s3)
df_symptom = get_dataset_df_str(dataset_paths, "dataset_symptomlog", s3)
df_enrolled = get_dataset_df_str(dataset_paths, "dataset_enrolledparticipants", s3)

dataset: recover-main-project/main/archive/2023-11-10/dataset_fitbitdailydata
Index(['ParticipantIdentifier', 'Date', 'ActivityCalories', 'BodyBmi',
       'BodyFat', 'BodyWeight', 'BreathingRate', 'CardioScore', 'Calories',
       'CaloriesBMR', 'Distance', 'Elevation', 'Floors', 'FoodCaloriesIn',
       'HeartRateIntradayCount', 'HeartRateIntradayMinuteCount',
       'HeartRateZone_OutOfRangeCaloriesOut', 'HeartRateZone_OutOfRangeMax',
       'HeartRateZone_OutOfRangeMin', 'HeartRateZone_OutOfRangeMinutes',
       'HeartRateZone_FatBurnCaloriesOut', 'HeartRateZone_FatBurnMax',
       'HeartRateZone_FatBurnMin', 'HeartRateZone_FatBurnMinutes',
       'HeartRateZone_CardioCaloriesOut', 'HeartRateZone_CardioMax',
       'HeartRateZone_CardioMin', 'HeartRateZone_CardioMinutes',
       'HeartRateZone_PeakCaloriesOut', 'HeartRateZone_PeakMax',
       'HeartRateZone_PeakMin', 'HeartRateZone_PeakMinutes', 'Hrv_DailyRmssd',
       'Hrv_DeepRmssd', 'MinutesFairlyActive', 'MinutesLightlyActive'

In [5]:
df_sleep_logs = get_dataset_df_str(dataset_paths, "dataset_fitbitsleeplogs", s3)
df_sleep_logs['Date'] = df_sleep_logs['EndDate'].apply(lambda x: pd.to_datetime(str(x).split("T")[0]))
# df_sleep_logs = get_sleep_date(df_sleep_logs, "StartDate", "EndDate", "Date")
df_sleep_logs_detail = get_dataset_df_str(dataset_paths, "dataset_fitbitsleeplogdetails", s3)
# df_intraday = get_dataset_df_str(dataset_paths, "dataset_fitbitintradaycombined", s3)
df_symptomval = get_dataset_df_str(dataset_paths, "dataset_symptomlog_value_treatments", s3)

dataset: recover-main-project/main/archive/2023-11-10/dataset_fitbitsleeplogs
Index(['ParticipantIdentifier', 'StartDate', 'EndDate', 'Duration',
       'Efficiency', 'InfoCode', 'MinutesAfterWakeup', 'MinutesAsleep',
       'MinutesAwake', 'MinutesToFallAsleep', 'TimeInBed', 'Type',
       'SleepLevelDeep', 'SleepLevelLight', 'SleepLevelRem', 'SleepLevelWake',
       'SleepLogDetails', 'LogId', 'ParticipantID', 'export_start_date',
       'export_end_date', 'SleepLevelAwake', 'SleepLevelAsleep',
       'SleepLevelRestless', 'IsMainSleep', 'LogType', 'ModifiedDate',
       'cohort'],
      dtype='object')
dataset: recover-main-project/main/archive/2023-11-10/dataset_symptomlog_value_treatments
Index(['id', 'index', 'Value_treatments_id', 'DataPointKey', 'cohort'], dtype='object')
dataset: recover-main-project/main/archive/2023-11-10/dataset_symptomlog_value_treatments
Index(['id', 'index', 'Value_treatments_id', 'DataPointKey', 'cohort'], dtype='object')


In [6]:
df_sleep_logs

,ParticipantIdentifier,StartDate,EndDate,Duration,Efficiency,InfoCode,MinutesAfterWakeup,MinutesAsleep,MinutesAwake,MinutesToFallAsleep,...,export_start_date,export_end_date,SleepLevelAwake,SleepLevelAsleep,SleepLevelRestless,IsMainSleep,LogType,ModifiedDate,cohort,Date
0,RA12003-00055,2019-06-16T20:59:00,2019-06-17T07:06:00,36420000,88,0,3,511,96,0,...,2023-07-21T00:00:00,2023-07-22T00:00:00,None,None,None,None,None,None,adults_v1,2019-06-17
1,RA12501-00258,2019-07-02T22:00:30,2019-07-03T05:53:00,28320000,95,0,0,419,53,0,...,2023-10-10T00:00:00,2023-10-11T00:00:00,None,None,None,None,None,None,adults_v1,2019-07-03
2,RA11306-00032,2019-08-03T23:19:00,2019-08-04T08:08:00,31740000,88,0,0,452,77,0,...,2023-08-31T00:00:00,2023-09-01T00:00:00,None,None,None,None,None,None,adults_v1,2019-08-04
3,RA12003-00055,2019-09-10T23:58:00,2019-09-11T05:52:30,21240000,96,3,3,338,13,0,...,2023-07-21T00:00:00,2023-07-22T00:00:00,0,338,16,None,None,None,adults_v1,2019-09-11
4,RA11101-00251,2019-10-06T22:51:00,2019-10-07T05:58:00,25620000,93,0,0,374,53,0,...,2023-05-21T00:00:00,2023-05-23T00:00:00,None,None,None,None,None,None,adults_v1,2019-10-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182211,RP25702-00152,2022-10-19T10:10:30,2022-10-19T16:53:30,24180000,47,0,0,343,60,0,...,2023-09-29T00:00:00,2023-09-30T00:00:00,None,None,None,True,auto_detected,None,pediatric_v1,2022-10-19
182212,RP25702-00001,2023-10-04T01:29:30,2023-10-04T08:22:00,24720000,95,0,0,356,56,0,...,2023-10-12T00:00:00,2023-10-13T00:00:00,None,None,None,True,auto_detected,None,pediatric_v1,2023-10-04
182213,RP25702-00022,2023-10-12T23:35:00,2023-10-13T07:04:30,26940000,89,0,7,376,73,0,...,2023-10-21T00:00:00,2023-10-22T00:00:00,None,None,None,True,auto_detected,None,pediatric_v1,2023-10-13
182214,RP25702-00001,2023-10-19T04:00:00,2023-10-19T07:55:30,14100000,92,0,0,207,28,0,...,2023-10-27T00:00:00,2023-10-28T00:00:00,None,None,None,True,auto_detected,2023-10-27T14:21:02Z,pediatric_v1,2023-10-19


In [7]:
df = update_df_object_date(df, ['Date'])

In [14]:
merged_df_sleep = df.merge(df_sleep_logs, how="inner", on=['ParticipantIdentifier', 'Date'])

In [15]:
print(df.describe())

                                Date
count                         537660
mean   2023-02-12 21:51:46.983595776
min              2009-01-01 00:00:00
25%              2022-12-16 00:00:00
50%              2023-05-08 00:00:00
75%              2023-08-27 00:00:00
max              2023-11-08 00:00:00


In [16]:
print(df_sleep_logs.describe())

       SleepLogDetails                           Date
count    182202.000000                         182216
mean      91101.500000  2023-03-24 17:34:32.248320768
min           1.000000            2019-04-06 00:00:00
25%       45551.250000            2022-12-20 00:00:00
50%       91101.500000            2023-06-07 00:00:00
75%      136651.750000            2023-09-13 00:00:00
max      182202.000000            2023-11-08 00:00:00
std       52597.331211                            NaN


In [18]:
print(merged_df_sleep.describe())

                                Date  SleepLogDetails
count                         181984    181970.000000
mean   2023-03-25 02:23:54.587655936     91104.888124
min              2019-04-06 00:00:00         1.000000
25%              2022-12-21 00:00:00     45556.250000
50%              2023-06-08 00:00:00     91103.500000
75%              2023-09-14 00:00:00    136655.750000
max              2023-11-08 00:00:00    182202.000000
std                              NaN     52597.043649


In [25]:
print(len(df['ParticipantIdentifier'].unique()))
print(len(df_sleep_logs['ParticipantIdentifier'].unique()))
print(len(merged_df_sleep['ParticipantIdentifier'].unique()))

2128
1468
1468


In [ ]:
plt.scatter(merged_df_sleep['TimeInBed'],df_sleep_ad['daily_adherence_level'])
plt.xlabel("Average Time In Bed")
plt.ylabel("Average Daily Adherence Level")

In [26]:
merged_df_sleep.columns

Index(['ParticipantIdentifier', 'Date', 'ActivityCalories', 'BodyBmi',
       'BodyFat', 'BodyWeight', 'BreathingRate', 'CardioScore', 'Calories',
       'CaloriesBMR', 'Distance', 'Elevation', 'Floors', 'FoodCaloriesIn',
       'HeartRateIntradayCount', 'HeartRateIntradayMinuteCount',
       'HeartRateZone_OutOfRangeCaloriesOut', 'HeartRateZone_OutOfRangeMax',
       'HeartRateZone_OutOfRangeMin', 'HeartRateZone_OutOfRangeMinutes',
       'HeartRateZone_FatBurnCaloriesOut', 'HeartRateZone_FatBurnMax',
       'HeartRateZone_FatBurnMin', 'HeartRateZone_FatBurnMinutes',
       'HeartRateZone_CardioCaloriesOut', 'HeartRateZone_CardioMax',
       'HeartRateZone_CardioMin', 'HeartRateZone_CardioMinutes',
       'HeartRateZone_PeakCaloriesOut', 'HeartRateZone_PeakMax',
       'HeartRateZone_PeakMin', 'HeartRateZone_PeakMinutes', 'Hrv_DailyRmssd',
       'Hrv_DeepRmssd', 'MinutesFairlyActive', 'MinutesLightlyActive',
       'MinutesSedentary', 'MinutesVeryActive', 'RestingHeartRate', 'SpO2_Av

In [40]:
df_sleep_logs_detail

,id,index,Type,StartDate,EndDate,Value,LogId,cohort
0,26,0,ShortSleepLevel,2021-03-04T06:25:00,2021-03-04T06:25:30,wake,31182907062,adults_v1
1,26,1,ShortSleepLevel,2021-03-04T06:20:00,2021-03-04T06:23:00,wake,31182907062,adults_v1
2,26,2,ShortSleepLevel,2021-03-04T06:15:00,2021-03-04T06:15:30,wake,31182907062,adults_v1
3,26,3,ShortSleepLevel,2021-03-04T06:10:30,2021-03-04T06:12:00,wake,31182907062,adults_v1
4,26,4,ShortSleepLevel,2021-03-04T05:51:30,2021-03-04T05:52:00,wake,31182907062,adults_v1
...,...,...,...,...,...,...,...,...
6539186,135675,50,SleepLevel,2023-10-29T06:20:30,2023-10-29T06:24:00,wake,43291306486,pediatric_v1
6539187,135675,51,SleepLevel,2023-10-29T05:30:00,2023-10-29T06:20:30,light,43291306486,pediatric_v1
6539188,135675,52,SleepLevel,2023-10-29T04:45:30,2023-10-29T05:30:00,deep,43291306486,pediatric_v1
6539189,135675,53,SleepLevel,2023-10-29T04:34:00,2023-10-29T04:45:30,light,43291306486,pediatric_v1


In [41]:
df

,ParticipantIdentifier,Date,ActivityCalories,BodyBmi,BodyFat,BodyWeight,BreathingRate,CardioScore,Calories,CaloriesBMR,...,Tracker_MinutesLightlyActive,Tracker_MinutesSedentary,Tracker_MinutesVeryActive,Tracker_Steps,Water,ModifiedDate,ParticipantID,export_start_date,export_end_date,cohort
0,RA11001-00011,2023-03-18T00:00:00,0,25.264387130737305,0.0,73.1,None,None,1680,1680,...,0,1440,0,0,0.0,2023-09-27T09:18:00Z,F83EFD9C-6EE6-47F2-BB68-E9C9310741EC,2023-09-27T00:00:00,2023-09-28T00:00:00,adults_v1
1,RA11001-00011,2023-06-19T00:00:00,0,25.23224639892578,0.0,73.007,None,None,1679,1679,...,0,1440,0,0,0.0,2023-09-27T09:17:59Z,F83EFD9C-6EE6-47F2-BB68-E9C9310741EC,2023-09-27T00:00:00,2023-09-28T00:00:00,adults_v1
2,RA11001-00041,2023-09-11T00:00:00,0,21.991069793701172,0.0,73.485,None,52-56,1719,1719,...,0,1440,0,0,0.0,2023-10-06T02:05:40Z,BA1757D5-999C-472D-8C53-82BE59FE6529,2023-10-06T00:00:00,2023-10-07T00:00:00,adults_v1
3,RA11001-00056,2022-11-14T00:00:00,0,24.658823013305664,0.0,71.264,None,None,1495,1495,...,0,1440,0,0,0.0,2023-10-11T06:24:29Z,4038850E-F9F8-4AEA-977F-0AD756966315,2023-10-11T00:00:00,2023-10-12T00:00:00,adults_v1
4,RA11001-00075,2023-10-21T00:00:00,287,20.32269859313965,0.0,50.349,None,38-42,1282,1065,...,135,1305,0,2421,0.0,2023-10-29T05:59:12Z,EBB6CF8D-9116-49F6-83C8-E0D9A079C4AA,2023-10-29T00:00:00,2023-10-30T00:00:00,adults_v1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
537655,RP25702-00152,2023-10-10T00:00:00,997,None,None,None,17.4,32-36,2216,1412,...,288,738,0,7872,None,2023-10-18T10:28:56Z,E3DE4A7D-1298-4E1C-B863-7C1D54EB37D2,2023-10-18T00:00:00,2023-10-19T00:00:00,pediatric_v1
537656,RP25702-00215,2023-05-22T00:00:00,486,25.700578689575195,39.0,61.666,None,36,1702,1315,...,107,1298,20,5606,0.0,2023-10-19T15:23:48Z,6ED88D40-59FA-4AA5-9467-F4E5E4FEC9C2,2023-10-19T00:00:00,2023-10-20T00:00:00,pediatric_v1
537657,RP25702-00215,2023-06-12T00:00:00,501,25.884374618530273,39.0,62.107,None,33-37,1680,1316,...,184,1256,0,4318,0.0,2023-10-19T15:23:48Z,6ED88D40-59FA-4AA5-9467-F4E5E4FEC9C2,2023-10-19T00:00:00,2023-10-20T00:00:00,pediatric_v1
537658,RP25702-00215,2023-08-25T00:00:00,0,26.390750885009766,39.0,63.322,None,32-36,1311,1311,...,0,1440,0,0,0.0,2023-10-19T15:23:49Z,6ED88D40-59FA-4AA5-9467-F4E5E4FEC9C2,2023-10-19T00:00:00,2023-10-20T00:00:00,pediatric_v1


In [5]:
df_sleep_logs['ParticipantIdentifier'].value_counts()

ParticipantIdentifier
RA11601-00313    1998
RA12501-00258    1736
RA12410-00021    1679
RA11601-00570    1508
RA12003-00055    1472
                 ... 
RA12003-00028       1
RA11702-00171       1
RA11302-00069       1
RA12004-00223       1
RA11601-00506       1
Name: count, Length: 1399, dtype: int64

In [6]:
df['ParticipantIdentifier'].value_counts()

ParticipantIdentifier
RA12305-00011    5416
RA11702-00063    1669
RA12410-00021    1666
RA12003-00055    1652
RA12501-00258    1651
                 ... 
RA12501-00164       1
RA11602-00062       1
RA11702-00181       1
RA12303-00075       1
RA11101-01032       1
Name: count, Length: 2054, dtype: int64

In [7]:
df_devices['ParticipantIdentifier']

0        RA11001-00033
1        RA11001-00041
2        RA11001-00120
3        RA11001-00240
4        RA11002-00035
             ...      
64610    RP25702-00215
64611    RP25702-00021
64612    RP25702-00021
64613    RP25702-00022
64614    RP25702-00215
Name: ParticipantIdentifier, Length: 64615, dtype: object

In [8]:
df['ParticipantIdentifier'].value_counts()

ParticipantIdentifier
RA12305-00011    5416
RA11702-00063    1669
RA12410-00021    1666
RA12003-00055    1652
RA12501-00258    1651
                 ... 
RA12501-00164       1
RA11602-00062       1
RA11702-00181       1
RA12303-00075       1
RA11101-01032       1
Name: count, Length: 2054, dtype: int64

In [8]:
# device_merge
df_with_device

NameError: name 'df_with_device' is not defined

In [9]:
device_merge = df_devices.iloc[:, [0, 2, 6]]

In [10]:
device_merge

,ParticipantIdentifier,Device,export_end_date
0,RA11003-00032,MobileTrack,2023-09-18T00:00:00
1,RA11003-00065,Sense 2,2023-08-30T00:00:00
2,RA11003-00065,Sense 2,2023-09-18T00:00:00
3,RA11003-00121,Charge 5,2023-08-15T00:00:00
4,RA11003-00125,Sense 2,2023-09-06T00:00:00
...,...,...,...
29150,RA12410-00056,Inspire 2,2023-08-06T00:00:00
29151,RA12412-00020,Sense 2,2023-09-14T00:00:00
29152,RA12501-00324,Sense 2,2023-09-14T00:00:00
29153,RA12610-00087,Charge 5,2023-09-04T00:00:00


In [31]:
device_merge.insert(0, 'Device1', np.nan)

In [34]:
device_merge.loc[device_merge['ParticipantIdentifier']=='RA11003-00032', 'Device']

0        MobileTrack
982         Charge 5
1132        Charge 5
1436     MobileTrack
1572        Charge 5
            ...     
26691    MobileTrack
27280       Charge 5
27281    MobileTrack
27579       Charge 5
28309       Charge 5
Name: Device, Length: 82, dtype: object

In [27]:
device_merge.iloc[:,0].values

array(['RA11003-00032', 'RA11003-00065', 'RA11003-00065', ...,
       'RA12501-00324', 'RA12610-00087', 'RA12617-00002'], dtype=object)

In [20]:
aa = device_merge[device_merge['ParticipantIdentifier']=="RA11003-00065"]['Device'].value_counts()

In [28]:
aa.index[0]

'Sense 2'

In [10]:
df_with_device = pd.merge(df, device_merge, how="left", on=["ParticipantIdentifier", "export_end_date"])
df_with_device['HeartRateIntradayMinuteCount'] = df_with_device['HeartRateIntradayMinuteCount'].astype('float')

In [11]:
df_with_device['Device'].unique()

array([nan, 'Sense 2', 'Inspire HR', 'Aria 2', 'MobileTrack', 'Sense',
       'Charge 5', 'Luxe', 'Charge 2', 'Charge 3', 'Versa 2', 'Inspire',
       'Inspire 2', 'Charge 4', 'Versa 3', 'Versa Lite', 'Blaze',
       'Charge HR', 'Alta HR', 'Versa 4', 'Inspire 3', 'Aria',
       'Google Pixel Watch', 'Zip', 'Flex', 'Ionic', 'Flex 2', 'Versa',
       'Aria Air', 'Alta', 'Charge 6', 'Charge', 'Ace', 'Ace 2', 'Surge',
       'Classic', 'One'], dtype=object)

In [12]:
rate_list = []
effect_len_list = []
name_list = str(df_with_device['Device'].unique())
for device in df_with_device['Device'].unique():
    if str(device)=="nan":
        continue
    print(str(device))
    extracted_df = df_with_device[df_with_device['Device']==device]
    total_len = len(extracted_df)
    effect_len = len(extracted_df[extracted_df['HeartRateIntradayMinuteCount']>-1])
    print(" effect len : " + str(effect_len))
    effect_len_list.append(effect_len)
    print(" effective rate: " + str(effect_len / total_len))
    rate_list.append(effect_len / total_len)

Sense 2
 effect len : 52173
 effective rate: 0.8272761868518694
Inspire HR
 effect len : 345
 effective rate: 0.17037037037037037
Aria 2
 effect len : 1335
 effective rate: 0.3239504974520747
MobileTrack
 effect len : 10653
 effective rate: 0.6264259673056568
Sense
 effect len : 1380
 effective rate: 0.19765110283586365
Charge 5
 effect len : 13518
 effective rate: 0.5534266764922623
Luxe
 effect len : 641
 effective rate: 0.19477362503798237
Charge 2
 effect len : 746
 effective rate: 0.2938164631744781
Charge 3
 effect len : 339
 effective rate: 0.20433996383363473
Versa 2
 effect len : 1849
 effective rate: 0.19109136006614302
Inspire
 effect len : 192
 effective rate: 0.27311522048364156
Inspire 2
 effect len : 2834
 effective rate: 0.2715599846684553
Charge 4
 effect len : 1884
 effective rate: 0.22375296912114015
Versa 3
 effect len : 2115
 effective rate: 0.2266395199314188
Versa Lite
 effect len : 107
 effective rate: 0.097538742023701
Blaze
 effect len : 786
 effective rate: 0

In [106]:
effect_len_list

[23410,
 7441,
 501,
 1235,
 180,
 2300,
 2091,
 709,
 231,
 1504,
 453,
 1541,
 67,
 776,
 715,
 217,
 392,
 602,
 719,
 429,
 377,
 94,
 162,
 197,
 400,
 67,
 161,
 149,
 29,
 86,
 1,
 1,
 23,
 12,
 18]

In [100]:
# name_list[0]+name_list[1:]
name_list = str(df_with_device['Device'].unique())

In [108]:
name_list = ['Sense 2', 'Charge 5', 'Charge 2', 'Versa 2', 'Zip', 'Inspire 2', 'MobileTrack', 'Sense' ,'Inspire HR', 'Charge 4', 'Luxe' ,'Versa 3', 'Versa Lite', 'Blaze', 'Versa 4' ,'Charge HR', 'Inspire 3', 'Alta HR', 'Aria', 'Google Pixel Watch', 'Aria 2' ,'Inspire', 'Flex', 'Charge 3', 'Versa', 'Flex 2', 'Alta', 'Aria Air', 'Ionic' ,'Ace' ,'Ace 2' ,'Classic' ,'One' ,'Charge', 'Surge']

In [111]:
data = {'effective len': effect_len_list, 'effective rate': rate_list}
test_df = pd.DataFrame(data, index=name_list)



In [112]:
test_df

,effective len,effective rate
Sense 2,23410,0.812452
Charge 5,7441,0.512713
Charge 2,501,0.255092
Versa 2,1235,0.193392
Zip,180,0.369610
Inspire 2,2300,0.267753
MobileTrack,2091,0.365815
Sense,709,0.212022
Inspire HR,231,0.184652
Charge 4,1504,0.248185


In [87]:
# extracted_df[extracted_df['HeartRateIntradayMinuteCount']>-1]
for device in df_with_device['Device'].unique():
    print(str(device))


Sense 2
nan
Charge 5
Charge 2
Versa 2
Zip
Inspire 2
MobileTrack
Sense
Inspire HR
Charge 4
Luxe
Versa 3
Versa Lite
Blaze
Versa 4
Charge HR
Inspire 3
Alta HR
Aria
Google Pixel Watch
Aria 2
Inspire
Flex
Charge 3
Versa
Flex 2
Alta
Aria Air
Ionic
Ace
Ace 2
Classic
One
Charge
Surge


In [59]:
df_with_device[(df_with_device['Device']=='Sense 2')]

,ParticipantIdentifier,Date,ActivityCalories,BodyBmi,BodyFat,BodyWeight,BreathingRate,CardioScore,Calories,CaloriesBMR,...,Tracker_Steps,Water,ModifiedDate,ParticipantID,export_start_date,export_end_date,Elevation,Floors,cohort,Device
0,RA11003-00024,2023-07-16T00:00:00,0,35.08241653442383,0.0,89.811,None,None,1475,1475,...,0,0.0,2023-07-24T22:20:28Z,E9F0336B-8844-4971-B372-C4EA6A7523C5,2023-07-24T00:00:00,2023-07-25T00:00:00,0,0,adults_v1,Sense 2
10,RA11003-00106,2023-09-17T00:00:00,0,27.073375701904297,0.0,83.007,None,43-47,1224,1224,...,0,0.0,2023-09-17T20:26:11Z,AEB3826B-7114-436D-9B21-88CCF2A94279,2023-09-17T00:00:00,2023-09-18T00:00:00,0,0,adults_v1,Sense 2
27,RA11006-00032,2023-08-30T00:00:00,892,29.26958656311035,0.0,77.29,None,32-36,2194,1444,...,6912,0.0,2023-09-07T19:54:55Z,1DA1AF84-10BF-4B9E-814F-FD53642928F0,2023-09-07T00:00:00,2023-09-08T00:00:00,45.72,15,adults_v1,Sense 2
58,RA11101-00196,2023-08-11T00:00:00,922,22.74387550354004,0.0,73.936,None,51-55,2363,1580,...,8431,0.0,2023-08-19T21:52:45Z,6A0B0CF8-C585-4B4C-9E9C-F5085FC78AE8,2023-08-19T00:00:00,2023-08-20T00:00:00,18.29,6,adults_v1,Sense 2
70,RA11101-00261,2022-09-04T00:00:00,591,36.907752990722656,0.0,109.951,None,33-37,2540,1975,...,2734,0.0,2023-09-03T13:28:17Z,0986720D-0E3B-4419-85B0-3BF2D355D26F,2023-09-03T00:00:00,2023-09-04T00:00:00,3,1,adults_v1,Sense 2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
324347,RA12414-00055,2023-05-18T00:00:00,619,26.47917366027832,0.0,66.942,None,32-36,1616,1149,...,4464,0.0,2023-09-08T10:23:55Z,102BB97F-10D6-4F23-B633-C6AA2C53F331,2023-09-08T00:00:00,2023-09-09T00:00:00,9,3,adults_v1,Sense 2
324348,RA12414-00055,2023-05-21T00:00:00,1106,26.41034698486328,0.0,66.768,17.8,32-36,2019,1149,...,8788,0.0,2023-09-08T10:23:55Z,102BB97F-10D6-4F23-B633-C6AA2C53F331,2023-09-08T00:00:00,2023-09-09T00:00:00,27,9,adults_v1,Sense 2
324354,RA12501-00026,2023-08-30T00:00:00,1110,26.453205108642578,25.0,69.853,13.0,35-39,2300,1414,...,8433,0.0,2023-09-07T02:31:16Z,84DA23CD-A0F0-46E1-A2B4-69D4915560B9,2023-09-07T00:00:00,2023-09-08T00:00:00,9.14,3,adults_v1,Sense 2
324392,RA12610-00022,2023-08-08T00:00:00,1102,32.24666976928711,25.0,87.898,17.0,30-34,2323,1487,...,7517,0.0,2023-08-16T02:51:29Z,D6259357-D09E-4083-884D-8A3CA3039482,2023-08-16T00:00:00,2023-08-17T00:00:00,18.29,6,adults_v1,Sense 2
